In [15]:
import sys
sys.path.insert(0,'/home/ruthvik/ECE-697-Fall-2022')

In [25]:
import math
import torch
import numpy as np
import torchvision
import torch.nn as nn
import torch.nn.functional as F
from transformer.randomaug import RandAugment
import torchvision.transforms as transforms

import torchvision.transforms as transforms
from einops import rearrange, reduce, repeat
from transformer.Optim import ScheduledOptim
from transformer.randomaug import RandAugment
from einops.layers.torch import Rearrange, Reduce
from cosine_annealing_warmup import CosineAnnealingWarmupRestarts

In [20]:
print('PyTorch version:{}'.format(torch.__version__))
print('Is CUDA available:{}'.format(torch.cuda.is_available()))
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print('Device is{}'.format(device))

PyTorch version:1.12.1+cu113
Is CUDA available:True
Device iscuda:0


In [21]:
batch_size = 100
size = 32
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.Resize(size),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])


N = 2; M = 14;
transform_train.transforms.insert(0, RandAugment(N, M))
    
transform_test = transforms.Compose([
    transforms.Resize(size),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(root='/home/ruthvik/data/', train=True,
                                        download= True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle= True, num_workers=4)

testset = torchvision.datasets.CIFAR10(root='/home/ruthvik/data/', train=False,
                                       download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=4)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

Files already downloaded and verified
Files already downloaded and verified


In [22]:
class PatchEmbedding(nn.Module):
    def __init__(self, in_channels: int=3, patch_size: int=4, d_model: int=512, img_size: int=32,
                n_conv_layers: int=1):
        self.patch_size = patch_size
        super().__init__()
        # using a conv layer instead of a linear one -> performance gains
        # same_conv_layer means the shapes of input and output images is same as opposed to valid mode conv.
        self.same_conv_layer_stack = nn.ModuleList([nn.Conv2d(in_channels, in_channels, kernel_size=5, stride=1, padding=2) \
                                                    for i in range(n_conv_layers)])
        self.conv_proj_layer = nn.Conv2d(in_channels, d_model, kernel_size=patch_size, stride=patch_size)
        self.re_arrange_layer = Rearrange('b e (h) (w) -> b (h w) e')
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        #self.position_token = nn.Parameter(torch.randn((img_size // patch_size)**2 + 1, d_model))
        
                
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, *_ = x.shape
        for same_conv_layer in self.same_conv_layer_stack:
            x = same_conv_layer(x)
        convd_img = self.conv_proj_layer(x)
        #print('Output of convolution:{}'.format(convd_img.shape))
        re_arranged_ip = self.re_arrange_layer(convd_img)
        #print('Rearranged ip:{}'.format(re_arranged_ip.shape))
        cls_token = repeat(self.cls_token, '() n e -> b n e', b=b)
        #print('CLS token:{}'.format(cls_token.shape))
        concated_ip = torch.cat([cls_token, re_arranged_ip], axis=1)
        #concated_ip += self.position_token
        #print('Concated ip:{}'.format(concated_ip.shape))
        
        return concated_ip

In [26]:
src_img = torch.Tensor(np.random.randint(0, 255, size=(2,3, 32, 32)))
print('Original image:{}'.format(src_img.shape))
concated_ip = PatchEmbedding(n_conv_layers=2)(src_img)
print('projected and cls_toen concated image:{}'.format(concated_ip.shape))

Original image:torch.Size([2, 3, 32, 32])
projected and cls_toen concated image:torch.Size([2, 65, 512])


In [27]:
def get_patch_positions(img_size, patch_size):
    """
    Return a tensor of shape (num_patches, 2) with (row, col) indices
    for each patch in the patch grid.
    """
    H = img_size // patch_size
    W = img_size // patch_size
    positions = []
    for r in range(H):
        for c in range(W):
            positions.append([r, c])
    # positions is a list of length H*W, each [r, c]
    patch_positions = torch.tensor(positions)  # shape (H*W, 2)
    return patch_positions


In [29]:
# For 32x32 images, patch_size=4 -> 8x8=64 patches
base_positions = get_patch_positions(32, 4)  # shape (64,2) or get_patch_positions_in_original_space
B = concated_ip.shape[0]
positions_2d_batched = base_positions.unsqueeze(0).repeat(B, 1, 1)  # (B, 64, 2)
print(positions_2d_batched.shape)

torch.Size([2, 64, 2])


In [6]:
def compute_shape_bias_penalty(patch_positions, patch_embeddings, alpha=1.0, dist_scale=1.0):
    """
    patch_positions: (N, 2) array of (row, col) for each patch
    patch_embeddings: (N, d) array for patch embeddings (for correlation)
    alpha: scalar controlling penalty strength
    dist_scale: scalar controlling distance decay
    """
    # Number of patches
    N = patch_positions.shape[0]
    
    # -- (1) Compute pairwise correlations (e.g. cosine similarity) --
    # Normalize embeddings to get cos sim
    norms = patch_embeddings.norm(dim=1, keepdim=True) + 1e-6
    normalized = patch_embeddings / norms
    corr_matrix = normalized @ normalized.T  # (N, N)

    # -- (2) Compute pairwise distances (Euclidean or Manhattan, etc.) --
    row_diff = patch_positions[:, 0].unsqueeze(1) - patch_positions[:, 0].unsqueeze(0)
    col_diff = patch_positions[:, 1].unsqueeze(1) - patch_positions[:, 1].unsqueeze(0)
    dist_matrix = torch.sqrt(row_diff**2 + col_diff**2)  # (N, N)
    
    # Example distance weight function = 1 / (1 + dist*dist_scale)
    dist_weight = 1.0 / (1.0 + dist_matrix * dist_scale)

    # -- (3) Combine them with a scaling alpha --
    penalty = alpha * corr_matrix * dist_weight

    # -- (4) Set diagonal = 0.0 to avoid penalizing self-attention --
    diag_idx = torch.arange(N, device=patch_positions.device)
    penalty[diag_idx, diag_idx] = 0.0
    
    return penalty  # (N, N)

In [43]:
class ScaledDotProductAttention(nn.Module):
    ''' Scaled Dot-Product Attention with optional shape-bias penalty. '''

    def __init__(self, temperature, attn_dropout=0.1, alpha=1.0, dist_scale=1.0):
        super().__init__()
        self.temperature = temperature
        self.dropout = nn.Dropout(attn_dropout)
        self.alpha = alpha
        self.dist_scale = dist_scale

    def forward(
        self, 
        q, k, v,                 # q, k, v: (B, heads, N, d)
        mask=None,               # optional attention mask
        patch_positions=None,    # (B, N, 2) or (N, 2) if single-image
        patch_embeddings=None    # (B, N, d) or (N, d)
    ):
        """
        q, k, v : (B, H, N, D)
        patch_positions: for shape bias (B, N, 2) if you have a batch
        patch_embeddings: for shape bias (B, N, E) or (B, N, D)
        """

        # (B, H, N, d) x (B, H, d, N) -> (B, H, N, N)
        attn_logits = torch.matmul(q / self.temperature, k.transpose(2, 3))

        B, H, N, _ = q.shape

        # ---------------------------------------------------
        # (1) Optionally compute shape-bias penalty per batch
        # ---------------------------------------------------
        if patch_positions is not None and patch_embeddings is not None:
            # We'll assume patch_positions and patch_embeddings are batched
            # shapes: (B, N, 2), (B, N, D)
            # We'll produce a penalty matrix for each item in batch
            penalty_matrices = []
            for b_idx in range(B):
                # shape: (N,2) and (N,d)
                pos_b = patch_positions[b_idx]
                emb_b = patch_embeddings[b_idx]
                # compute penalty for this sample
                M = compute_shape_bias_penalty(
                    pos_b, emb_b, alpha=self.alpha, dist_scale=self.dist_scale
                )
                # M is (N, N)
                penalty_matrices.append(M.unsqueeze(0))  # -> (1, N, N)

            # Stack into (B, N, N) and expand heads: (B, 1, N, N) -> (B, H, N, N)
            penalty_full = torch.stack(penalty_matrices, dim=0)
            print(penalty_full.shape)  ##<---- 2x1x64x64
            #penalty_full = penalty_full.unsqueeze(1).expand(-1, H, -1, -1)
            penalty_full = penalty_full.expand(-1, H, -1, -1)
            
            # Subtract from attn_logits
            # attn_logits shape = (B, H, N+1, N+1)
            # penalty_full shape = (B, H, N, N)
            attn_logits[..., 1:, 1:] = attn_logits[..., 1:, 1:] - penalty_full
            #print(attn_logits.shape, penalty_full.shape)
            #attn_logits = attn_logits - penalty_full

        # -----------------------------------------------
        # (2) If you have a mask, apply it here
        # -----------------------------------------------
        if mask is not None:
            # mask shape is typically (B, 1, N, N) or (B, H, N, N), 
            # with 0/1 entries
            attn_logits = attn_logits.masked_fill(mask == 0, -1e9)

        # -----------------------------------------------
        # (3) Softmax over last dim
        # -----------------------------------------------
        attn = F.softmax(attn_logits, dim=-1)
        attn = self.dropout(attn)

        # -----------------------------------------------
        # (4) Multiply by v to get final output
        # -----------------------------------------------
        output = torch.matmul(attn, v)  # (B, H, N, D)

        return output  # or (output, attn) if you want the attention map

In [39]:

class MultiHeadAttention(nn.Module):
    ''' Multi-Head Attention module '''

    def __init__(self, n_head, d_model, d_k, d_v, dropout=0.1):
        super().__init__()

        self.n_head = n_head
        self.d_k = d_k
        self.d_v = d_v

        self.w_qs = nn.Linear(d_model, n_head * d_k, bias=False)
        self.w_ks = nn.Linear(d_model, n_head * d_k, bias=False)
        self.w_vs = nn.Linear(d_model, n_head * d_v, bias=False)
        self.fc = nn.Linear(n_head * d_v, d_model, bias=False)

        self.attention = ScaledDotProductAttention(temperature=d_k ** 0.5)

        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(d_model, eps=1e-6)


    def forward(self, q, k, v, mask=None, patch_positions=None, patch_embeddings=None):    
                                          # (B, N, 2) or (N, 2) if single-image

        d_k, d_v, n_head = self.d_k, self.d_v, self.n_head
        sz_b, len_q, len_k, len_v = q.size(0), q.size(1), k.size(1), v.size(1)

        residual = q

        # Pass through the pre-attention projection: b x lq x (n*dv) = (1 x 65 x 512)
        # Separate different heads: b x lq x n x dv
        #print(q.shape)
        #print(self.w_qs.weight.shape)
        q = self.w_qs(q).view(sz_b, len_q, n_head, d_k) #(1 x 65 x 512) --> (1 x 65 x 8 x 64)
        k = self.w_ks(k).view(sz_b, len_k, n_head, d_k) #(1 x 65 x 512) --> (1 x 65 x 8 x 64)
        v = self.w_vs(v).view(sz_b, len_v, n_head, d_v) #(1 x 65 x 512) --> (1 x 65 x 8 x 64)

        # Transpose for attention dot product: b x n x lq x dv = (1 x 8 x 65 x 64)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)

        if mask is not None:
            mask = mask.unsqueeze(1)   # For head axis broadcasting.

        #q, attn = self.attention(q, k, v, mask=mask) # (1 x 8 x 65 x 64) and (1 x 8 x 65 x 65)
        q  = self.attention(q, k, v, mask=mask, patch_positions=patch_positions, patch_embeddings=patch_embeddings) # (10 x 8 x 12 x 64) and (10 x 8 x 12 x 12)
        # Transpose to move the head dimension back: b x lq x n x dv
        # Combine the last two dimensions to concatenate all the heads together: b x lq x (n*dv)
        # q.transpose(1, 2) --> (1 x 65 x 8 x 64)
        q = q.transpose(1, 2).contiguous().view(sz_b, len_q, -1) #(1 x 65 x 512)
        q = self.dropout(self.fc(q)) ## (1 x 65 x 512) x (512 x 512) --> (1 x 65 x 512)
        q += residual

        q = self.layer_norm(q)

        #return q, attn
        return q


In [44]:
n_head = 8
n_layers = 6
d_k = 64
d_v = 64
batch = 1
d_inner = 512
d_model = 512
q = concated_ip
k = q
v = q
print('Input to the multi head attention:{}'.format(q.shape))
multi_head_block = MultiHeadAttention(n_head=n_head, d_model=d_model, d_k=d_k, d_v=d_v)
#q, self_attn = multi_head_block(q, k ,v)  
patch_embeddings = concated_ip[:, 1:, :]
q = multi_head_block(q, k ,v, patch_positions=positions_2d_batched, patch_embeddings=patch_embeddings)
#print('Shape of q:{}, Shape of self attention:{}'.format(q.shape, self_attn.shape))
print('Shape of q:{}'.format(q.shape))

Input to the multi head attention:torch.Size([2, 65, 512])
torch.Size([2, 1, 64, 64])
Shape of q:torch.Size([2, 65, 512])


In [45]:
class PositionwiseFeedForward(nn.Module):
    ''' A two-feed-forward-layer module '''

    def __init__(self, d_in, d_hid, dropout=0.1):
        super().__init__()
        self.w_1 = nn.Linear(d_in, d_hid) # position-wise
        self.w_2 = nn.Linear(d_hid, d_in) # position-wise
        self.layer_norm = nn.LayerNorm(d_in, eps=1e-6)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        residual = x
        x = self.w_2(F.relu(self.w_1(x)))
        x = self.dropout(x)
        x += residual

        x = self.layer_norm(x)

        return x


In [46]:
class EncoderLayer(nn.Module):
    ''' Compose with two layers '''

    def __init__(self, d_model, d_inner, n_head, d_k, d_v, dropout=0.1):
        super(EncoderLayer, self).__init__()
        self.slf_attn = MultiHeadAttention(n_head, d_model, d_k, d_v, dropout=dropout)
        self.pos_ffn = PositionwiseFeedForward(d_model, d_inner, dropout=dropout)

    def forward(self, enc_input, slf_attn_mask=None, patch_positions=None, patch_embeddings=None):
        #enc_output, enc_slf_attn = self.slf_attn(
        #    enc_input, enc_input, enc_input, mask=slf_attn_mask)
        enc_output = self.slf_attn(
            enc_input, enc_input, enc_input, mask=slf_attn_mask, patch_positions=positions_2d_batched, patch_embeddings=patch_embeddings)
        if(type(enc_output) == tuple):
            enc_output, enc_slf_attn = enc_output
        enc_output = self.pos_ffn(enc_output)
        #return enc_output, enc_slf_attn
        return enc_output

In [47]:
class Encoder(nn.Module):
    ''' A encoder model with self attention mechanism. '''

    def __init__(
            self, n_layers, n_head, d_k, d_v,
            d_model, d_inner, dropout=0.1, n_conv_layers=1):

        super().__init__()

        self.position_enc = PatchEmbedding(d_model=d_model, n_conv_layers=n_conv_layers)
        self.dropout = nn.Dropout(p=dropout)
        self.layer_stack = nn.ModuleList([EncoderLayer(d_model, d_inner, n_head, d_k, d_v, dropout=dropout) \
                                          for _ in range(n_layers)])
        self.layer_norm = nn.LayerNorm(d_model, eps=1e-6)
        self.d_model = d_model
        self.positions_2d = get_patch_positions(32, 4)

    def forward(self, input_image, return_attns=False):
        B = input_image.shape[0]

        enc_slf_attn_list = []

        # -- Forward
        patch_embedding = self.position_enc(input_image)
        enc_output = self.dropout(patch_embedding)
        enc_output = self.layer_norm(enc_output)
        positions_2d_batched = self.positions_2d.unsqueeze(0).repeat(B, 1, 1)  # (B, 64, 2)

        for enc_layer in self.layer_stack:
            #enc_output, enc_slf_attn = enc_layer(enc_output, slf_attn_mask=None)
            #enc_slf_attn_list += [enc_slf_attn] if return_attns else []
            enc_output = enc_layer(enc_output, slf_attn_mask=None, patch_positions=positions_2d_batched, patch_embeddings=patch_embedding)

        #if return_attns:
        #    return enc_output, enc_slf_attn_list
        return enc_output

In [48]:
n_head = 8
n_layers = 6
d_k = 64
d_v = 64
d_inner = 512
d_model = 512

In [49]:
src_img = torch.Tensor(np.random.randint(0, 255, size=(2,3, 32, 32)))
print('Original image:{}'.format(src_img.shape))
encoder = Encoder(n_layers=6, n_head=6, d_k=d_k, d_v=d_v, d_model=d_model, d_inner=d_inner, n_conv_layers=2)
enc_output = encoder(input_image=src_img)
print('Encoder output:{}'.format(enc_output.shape))

Original image:torch.Size([2, 3, 32, 32])
Encoder output:torch.Size([2, 65, 512])


In [52]:
class ClassificationHeadWithAvgPooling(nn.Module):
    def __init__(self, d_model: int = 512, n_classes: int = 10):
        super().__init__()
        self.reduction_layer = Reduce('b n e -> b e', reduction='mean')
        self.layer_norm = nn.LayerNorm(d_model) 
        self.linear_layer = nn.Linear(d_model, n_classes)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        reduced_encoder_op = self.reduction_layer(x)
        #print('Reduced encoder shape:{}'.format(reduced_encoder_op.shape))
        layer_normed_reduced = self.layer_norm(reduced_encoder_op)
        output = self.linear_layer(layer_normed_reduced)
        return output

In [53]:
classification_head = ClassificationHeadWithAvgPooling()
print('Input to the classifier:{}'.format(enc_output.shape))
classification_op = classification_head(enc_output)
print('Classification output:{}'.format(classification_op.shape))

Input to the classifier:torch.Size([2, 65, 512])
Classification output:torch.Size([2, 10])


In [55]:
class ViT(nn.Module):
    def __init__(self, n_layers, n_head, d_k, d_v, d_model, d_inner, n_classes, 
                dropout=0.1, n_conv_layers=1):
        super().__init__()
        self.encoder = Encoder(n_layers=n_layers, n_head=n_head, d_k=d_k, d_v=d_v, d_model=d_model, 
                               d_inner=d_inner, n_conv_layers=n_conv_layers)
        self.classifier_head = ClassificationHeadWithAvgPooling(d_model=d_model, n_classes=n_classes)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        #print(x.shape)
        #encoder_op = self.encoder(input_image=x)
        encoder_op = self.encoder(x)
        classifier_op = self.classifier_head(encoder_op)
        return classifier_op

In [56]:
n_head = 8
n_layers = 6
d_k = 64
d_v = 64
batch = 1
d_inner = 512
d_model = 512
n_classes= 10
n_conv_layers = 4
vit = ViT(n_layers=n_layers, n_head=n_head, d_k=d_k, d_v=d_v, d_model=d_model, d_inner=d_inner,
         n_classes=n_classes, n_conv_layers=n_conv_layers).to(device)
#vit = torch.nn.DataParallel(vit)

In [57]:
src_img = torch.Tensor(np.random.randint(0, 255, size=(2,3, 32, 32))).to(device)
print('Original image:{}'.format(src_img.shape))
vit_op = vit(src_img)
print('vit output:{}'.format(vit_op.shape))

Original image:torch.Size([2, 3, 32, 32])
vit output:torch.Size([2, 10])
